# POSG Validation — SQL + NoSQL (Phase 15A + 15B combined)

Combines `POSG_SQL-2.ipynb` and `POSG_NOSQL.ipynb` into one clean setup + test notebook.
Dead-end retries and two bugs from the originals are removed:
- NoSQL checkpoint symlinks were accidentally left pointing at the SQL checkpoints in one exploratory cell — fixed here.
- The per-database MongoDB loading cell hardcoded `"concert_singer"` regardless of the loop variable — replaced with `convert_all()` (loads all 166 databases correctly).

See `docs/phase15_posg_findings.md` for the full write-up of what these tests found.

**Status as of last run**: SQL dev-split (hard-only) EX 63.3% vs 60.0% greedy. NoSQL train-split EX 76.7% vs 73.3% greedy.
**Pending in this notebook**: SQL full-difficulty (non `--hard`) EX — the number comparable to the plan's >82% target.

## 0. Compute budget — pick T4, not A100

Every cell below is inference-only (no training) and both "full" smoke tests
sample `--n 30` questions, not the full 1034-question dev set — this notebook
is light. **T4 (~1.9 CU/hr) comfortably covers it**; A100 (~15 CU/hr, ~8x the
burn rate) buys no benefit here since nothing is training. Set
`Runtime > Change runtime type > T4 GPU` before running Section 1.

`GeneratorInfer` (`src/generator/infer.py`) now loads the 7B model in int8
(bitsandbytes) on any GPU under 24GB, so it fits entirely in VRAM on a T4
instead of getting silently offloaded to CPU — CPU offload was making a
single candidate take minutes instead of seconds. A100s (≥24GB) still load
full bf16.

The cell below reports which GPU you actually got and flags it if it's
something other than T4.

In [1]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        if total_gb >= 24:
            print("  -> >=24GB (A100-class). This notebook doesn't need it (inference-only, "
                  "n=30 samples) -- you're paying ~15 CU/hr for no speed benefit that "
                  "matters here. Runtime > Change runtime type > T4 GPU, then reconnect.")
        else:
            print("  -> <24GB (T4-class, ~1.9 CU/hr): right choice for this notebook. "
                  "GeneratorInfer loads the 7B model in int8 (bitsandbytes) so it fits "
                  "entirely in VRAM -- no CPU offload.")
else:
    print("No GPU detected -- Runtime > Change runtime type > T4 GPU.")

GPU 0: NVIDIA A100-SXM4-40GB  (39.5 GB)
  -> >=24GB (A100-class). This notebook doesn't need it (inference-only, n=30 samples) -- you're paying ~15 CU/hr for no speed benefit that matters here. Runtime > Change runtime type > T4 GPU, then reconnect.


## 1. Clone repo + install dependencies

In [2]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes

Cloning into 'Codegen'...
remote: Enumerating objects: 1128, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 1128 (delta 109), reused 122 (delta 50), pack-reused 931 (from 1)
Receiving objects: 100% (1128/1128), 17.78 MiB | 17.16 MiB/s, done.
Resolving deltas: 100% (828/828), done.
/content/Codegen
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.8 MB/s eta

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. If your Drive layout differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust `DRIVE` below.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

Mounted at /content/drive
lrwxrwxrwx 1 root root 58 Jul 19 07:13 models/generator_nosql -> /content/drive/MyDrive/codegen/checkpoints/generator_nosql
lrwxrwxrwx 1 root root 56 Jul 19 07:13 models/generator_sql -> /content/drive/MyDrive/codegen/checkpoints/generator_sql
lrwxrwxrwx 1 root root 52 Jul 19 07:13 models/sar_nosql -> /content/drive/MyDrive/codegen/checkpoints/sar_nosql
lrwxrwxrwx 1 root root 50 Jul 19 07:13 models/sar_sql -> /content/drive/MyDrive/codegen/checkpoints/sar_sql


In [4]:
# Safety net: force sar.backend to memory regardless of whether the phase/15-posg
# PR has been merged into main yet. ChromaDB's PersistentClient can't open an index
# over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

sar:
  # "memory" → SARRetriever: re-encodes corpus at startup (~30 sec). No ChromaDB needed.


## 3. Spider SQLite databases

Needed for SQL EX scoring, and as the source data for the MongoDB conversion below. Uploaded once as a zip to Drive (see `docs/phase15_posg_findings.md` for how it was built, from a sibling local project).

In [5]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

166


## 4. MongoDB setup (needed for NoSQL EX)

**Important**: `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine. `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod` if we don't clear them first — this bit us once already (see the findings doc). The `shutil.rmtree` below is required, not optional.

In [6]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(["mongod", "--dbpath", "/data/db", "--bind_ip", "127.0.0.1"])
time.sleep(5)
!mongosh --eval "db.version()"

deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
The following NEW packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
0 upgraded, 9 newly installed, 0 to remove and 145 not upgraded.
Need to get 189 MB of archives.
After this operation,

In [7]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

[1/166] academic — 15 collections, 42 fields
[2/166] activity_1 — 5 collections, 22 fields
[3/166] aircraft — 5 collections, 28 fields
[4/166] allergy_1 — 3 collections, 12 fields
[5/166] apartment_rentals — 6 collections, 31 fields
[6/166] architecture — 3 collections, 17 fields
[7/166] assets_maintenance — 14 collections, 64 fields
[8/166] baseball_1 — 26 collections, 352 fields
[9/166] battle_death — 3 collections, 18 fields
[10/166] behavior_monitoring — 11 collections, 64 fields
[11/166] bike_1 — 4 collections, 46 fields
[12/166] body_builder — 2 collections, 11 fields
[13/166] book_2 — 2 collections, 9 fields
[14/166] browser_web — 3 collections, 11 fields
[15/166] candidate_poll — 2 collections, 14 fields
[16/166] car_1 — 6 collections, 23 fields
[17/166] chinook_1 — 11 collections, 64 fields
[18/166] cinema — 3 collections, 17 fields
[19/166] city_record — 4 collections, 27 fields
[20/166] climbing — 2 collections, 12 fields
[21/166] club_1 — 3 collections, 15 fields
[22/166] c

In [8]:
# Verify real data landed (not just schema cache) before trusting any EX result
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print("formula_1.drivers count:", client['formula_1']['drivers'].count_documents({}))   # must be > 0

169 ['academic', 'activity_1', 'admin', 'aircraft', 'allergy_1', 'apartment_rentals', 'architecture', 'assets_maintenance', 'baseball_1', 'battle_death']
formula_1.drivers count: 842


## 5. SQL track — full dev-set smoke test (all difficulty)

`sql_dev_eval_full.json` (1034 Spider dev questions, real SchemaLinker `key_fields`) was rebuilt locally on Mac after the Colab runtime that originally built it disconnected. Upload it to Drive from your Mac first, then pull it in here. This run (no `--hard` filter) is the one comparable to the plan's >82% EX target — we only have the hard-only number (63.3%) so far.

In [10]:
%%time
# Timed so you can extrapolate to the full 1034-question dev set before
# committing compute-unit budget to it: (this cell's wall time / 30) * 1034.
!cp /content/drive/MyDrive/codegen/sql_dev_eval_full.json Data/cot_data/sql_dev_eval_full.json
!python -m scripts.run_posg_sql --smoke_test --n 30 --data Data/cot_data/sql_dev_eval_full.json

cp: cannot stat '/content/drive/MyDrive/codegen/sql_dev_eval_full.json': No such file or directory
Running on: cuda
Data source: Data/cot_data/sql_dev_eval_full.json
Loading SAR retriever ...
config.json: 100% 779/779 [00:00<00:00, 4.90MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.55MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 85.6MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 139MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 640kB/s]

model.safetensors: downloading bytes:  11% 144M/1.34G [00:01<00:07, 162MB/s, 9.63MB/s  ]
model.safetensors: downloading bytes:  26% 344M/1.34G [00:02<00:02, 387MB/s, 27.1MB/s  ]
model.safetensors: downloading bytes:  30% 398M/1.34G [00:02<00:02, 427MB/s, 32.0MB/s  ]
model.safetensors: reconstructing file:  35% 469M/1.34G [00:02<00:03, 266MB/s, 31.6MB/s  ]
model.safetensors: downloading bytes:  48% 649M/1.34G [00:02<00:01, 521MB/s, 53.2MB/s  ]
model.safetensors: reconstructing file:  65% 872M/1.34G [00:02<00:00, 477MB/s, 60.9MB

## 6. NoSQL track — train-split smoke test (all difficulty)

Already validated: EX 76.7% (POSG) vs 73.3% (greedy) on this exact command. Re-run here to confirm reproducibility in a fresh session, or change `--n`/`--seed` for a different sample.

In [11]:
%%time
!python -m scripts.run_posg_nosql --smoke_test --n 30

Running on: cuda
Data source: Data/cot_data/nosql_cot_train.json
Mongo URI: mongodb://localhost:27017  (make sure mongod is running and target DBs are loaded)
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2510.45it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 197.07it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 32.08it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.25it/s] 

[1/30] show the titles, and authors or editors for all books made after the year 1989.
  gold:     {"collection": "book_club", "pipeline": [{"$match": {"Year": {"$gt": 1989}}}, {"$project": {"book_title": "$Book_Title", "author_or_editor": "$Author_or_Editor", "_id": 0}}]}
  greedy:   {"collection": "book_club", "pipeline": [{"$match": {"Year": {"$gt": 1989}}}, {"$project": {"book_title": "$Book_Title", "author_or_editor": "$Author_or_Ed

## 7. Optional — DeepSeek API key

Only needed if you want to rebuild a dev-eval-set file (`scripts/build_dev_eval_set.py`) or run the `--question` single-question mode — the smoke tests above reuse pre-computed `key_fields` and never call DeepSeek. **Never commit this cell with a real key filled in.**

In [12]:
with open('.env', 'w') as f:
    f.write('DEEPSEEK_API_KEY=your_actual_key_here\n')

## 8. Free the GPU when you're done

Units keep burning as long as the runtime stays connected, even idle. Run
this once you've read the output above and don't need the session anymore —
it disconnects and releases the GPU. (Not run automatically: you may still
want to inspect variables or re-run a cell first.)

In [12]:
from google.colab import runtime
runtime.unassign()